# fp32 vs fp16 vs int8

Generation speed and memory for the same model at three precisions. fp16 and int8 need a CUDA GPU (int8 goes through bitsandbytes). On CPU, only fp32 runs.

Change `MODEL_NAME` to try something else. Each mode reloads the model from scratch, so a bigger model takes longer.

In [ ]:
import gc
import time

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "facebook/opt-350m"
PROMPT = "The history of machine learning begins"
NEW_TOKENS = 128
RUNS = 3

cuda = torch.cuda.is_available()
device = "cuda" if cuda else "cpu"
gpu_name = torch.cuda.get_device_name(0) if cuda else "cpu"
print("running on:", gpu_name)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
def sync():
    if cuda:
        torch.cuda.synchronize()


def load(mode):
    kwargs = {}
    if mode == "fp32":
        kwargs["torch_dtype"] = torch.float32
    elif mode == "fp16":
        kwargs["torch_dtype"] = torch.float16
    elif mode == "int8":
        kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        kwargs["device_map"] = {"": 0}
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **kwargs)
    if mode != "int8":
        model = model.to(device)
    return model.eval()


def bench(mode):
    gc.collect()
    if cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model = load(mode)
    weights_mb = model.get_memory_footprint() / 1e6
    inputs = tok(PROMPT, return_tensors="pt").to(model.device)

    def run():
        with torch.no_grad():
            model.generate(**inputs, max_new_tokens=NEW_TOKENS, min_new_tokens=NEW_TOKENS,
                           do_sample=False, pad_token_id=tok.eos_token_id)

    run()  # warmup, the first call is always slow
    times = []
    for _ in range(RUNS):
        sync()
        start = time.perf_counter()
        run()
        sync()
        times.append(time.perf_counter() - start)

    median = sorted(times)[len(times) // 2]
    result = {
        "mode": mode,
        "weights_mb": round(weights_mb),
        "peak_gpu_mb": round(torch.cuda.max_memory_allocated() / 1e6) if cuda else None,
        "seconds": round(median, 2),
        "tokens_per_sec": round(NEW_TOKENS / median, 1),
    }
    del model
    gc.collect()
    if cuda:
        torch.cuda.empty_cache()
    return result

In [ ]:
modes = ["fp32", "fp16", "int8"] if cuda else ["fp32"]
rows = []
for mode in modes:
    print("benchmarking", mode, "...")
    try:
        rows.append(bench(mode))
    except Exception as e:
        print(f"  {mode} failed: {e}")

df = pd.DataFrame(rows)
df

In [ ]:
import os
os.makedirs("results", exist_ok=True)
tag = gpu_name.replace(" ", "_").replace("/", "_")
df.assign(model=MODEL_NAME, gpu=gpu_name).to_csv(f"results/quantization_{tag}.csv", index=False)
print("saved results/quantization_" + tag + ".csv")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
df.plot.bar(x="mode", y="tokens_per_sec", ax=axes[0], legend=False, rot=0, title="Tokens per second")
df.plot.bar(x="mode", y="weights_mb", ax=axes[1], legend=False, rot=0, title="Weights (MB)")
for ax in axes:
    ax.set_xlabel("")
plt.tight_layout()
plt.show()

## Notes to self

- int8 through bitsandbytes is often *slower* than fp16 at small sizes, because of the dequantize overhead. It saves memory, but that doesn't always mean faster.
- Check peak memory as well as weight size. Activations and the KV cache add to it.
- Numbers depend a lot on the GPU. Don't compare a CSV from one card against another card.